# Armenian surname generator (character-level MLP)

This notebook trains a small neural network that predicts the next letter of an Armenian surname, then samples new surnames one letter at a time.

Pipeline: load and clean the surname list, build a letter vocabulary, turn each surname into (prefix, next letter) training examples, train a feed-forward network (embedding layer plus MLP), search a small hyperparameter grid, and generate surnames from an optional prefix.

Input data: `../data/surnames_hy.txt`, one surname per line, in Armenian script.

In [2]:
import copy
from collections import Counter
from itertools import product
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
BATCH_SIZE = 32
DATA_PATH = Path("../data/surnames_hy.txt")
MODEL_PATH = Path("best_surname_model.pth")

np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


## 1. Load and clean the data

Surnames are lowercased, empty lines are dropped, and duplicates are removed. The result is sorted so the order is the same on every run. (Iterating over a plain `set` gives a different order in each Python session, which would change the train/validation split even with a fixed seed.)

In [3]:
with open(DATA_PATH, encoding="utf-8") as f:
    raw_surnames = [line.strip() for line in f]

surnames = sorted({s.lower() for s in raw_surnames if s})

lengths = [len(s) for s in surnames]
print(f"Lines in file: {len(raw_surnames)} | unique surnames: {len(surnames)}")
print(f"Surname length: min {min(lengths)}, max {max(lengths)}")
print("Examples:", surnames[:5])

Lines in file: 689 | unique surnames: 689
Surname length: min 5, max 14
Examples: ['աբազյան', 'աբալակով', 'աբալյան', 'աբայան', 'աբաջյան']


## 2. Vocabulary

The vocabulary is the 39 Armenian characters plus three special tokens: `<START>` begins every surname, `<END>` ends it, and `<PAD>` fills unused positions. The letter "ու" is stored as two characters in the data (ո followed by ւ), so ւ is a token of its own.

The check below confirms that every character in the data is in the vocabulary and lists any vocabulary letters that never occur in the data.

In [4]:
ARMENIAN_LETTERS = [
    "ա", "բ", "գ", "դ", "ե", "զ", "է", "ը", "թ",
    "ժ", "ի", "լ", "խ", "ծ", "կ", "հ", "ձ", "ղ",
    "ճ", "մ", "յ", "ն", "շ", "ո", "չ", "պ", "ջ",
    "ռ", "ս", "վ", "տ", "ր", "ց", "ւ", "փ", "ք",
    "օ", "ֆ", "և",
]
SPECIAL_TOKENS = ["<START>", "<END>", "<PAD>"]

vocab = SPECIAL_TOKENS + ARMENIAN_LETTERS
char_to_id = {ch: i for i, ch in enumerate(vocab)}
id_to_char = {i: ch for ch, i in char_to_id.items()}
vocab_size = len(vocab)
START_ID, END_ID, PAD_ID = (char_to_id[t] for t in SPECIAL_TOKENS)

letter_counts = Counter(ch for s in surnames for ch in s)
unexpected = set(letter_counts) - set(ARMENIAN_LETTERS)
missing = set(ARMENIAN_LETTERS) - set(letter_counts)
assert not unexpected, f"Characters in the data but not in the vocabulary: {unexpected}"

print("Vocabulary size:", vocab_size)
print("Vocabulary letters that never occur in the data:", sorted(missing))
print("Most common letters:", letter_counts.most_common(10))

Vocabulary size: 42
Vocabulary letters that never occur in the data: ['ը']
Most common letters: [('ա', 1511), ('ն', 903), ('յ', 710), ('ր', 284), ('ո', 276), ('ի', 173), ('մ', 167), ('ե', 160), ('ս', 151), ('լ', 149)]


## 3. Encoding and training examples

Each surname becomes `[<START>, letters..., <END>]`. From a sequence of `n` tokens, `n - 1` training examples are built: the input is the first `i` tokens and the target is token `i`. Every input is padded on the right with `<PAD>` to a fixed length (`CONTEXT_LENGTH`, the longest surname plus two). Generation later builds its inputs the same way, so the model sees the same kind of input at training and sampling time.

In [5]:
CONTEXT_LENGTH = max(len(s) for s in surnames) + 2  # +2 for <START> and <END>


def encode_surname(surname):
    return [START_ID] + [char_to_id[ch] for ch in surname] + [END_ID]


def pad_right(ids, length=CONTEXT_LENGTH):
    return ids + [PAD_ID] * (length - len(ids))


def make_examples(names):
    """Turn surnames into (padded prefix, next token) pairs."""
    X, y = [], []
    for name in names:
        ids = encode_surname(name)
        for i in range(1, len(ids)):
            X.append(pad_right(ids[:i]))
            y.append(ids[i])
    return np.array(X), np.array(y)


# Sanity check: encode/decode round trip, and one training example
demo = "սարգսյան"
assert "".join(id_to_char[i] for i in encode_surname(demo)[1:-1]) == demo
X_demo, y_demo = make_examples([demo])
print([id_to_char[i] for i in X_demo[2]], "->", id_to_char[y_demo[2]])

['<START>', 'ս', 'ա', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>'] -> ր


## 4. Train/validation split

The split is made on surnames, before any examples are built (80% train, 20% validation).

In [6]:
rng = np.random.default_rng(SEED)
order = rng.permutation(len(surnames))
split = int(0.8 * len(surnames))
train_names = [surnames[i] for i in order[:split]]
val_names = [surnames[i] for i in order[split:]]

X_train, y_train = make_examples(train_names)
X_val, y_val = make_examples(val_names)
print(f"Surnames: train {len(train_names)}, validation {len(val_names)}")
print(f"Examples: train {X_train.shape}, validation {X_val.shape}")


def to_loader(X, y, shuffle):
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)
    )
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle)


train_loader = to_loader(X_train, y_train, shuffle=True)
val_loader = to_loader(X_val, y_val, shuffle=False)

Surnames: train 551, validation 138
Examples: train (5203, 16), validation (1327, 16)


## 5. Model

The model embeds every position of the padded prefix, flattens the embeddings into one vector, and passes it through fully connected layers. The output is one raw score (logit) per vocabulary token. `padding_idx` keeps the `<PAD>` embedding at zero so padding carries no signal.

In [7]:
class SurnameGeneratorModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_layers,
                 context_length, dropout_rate, activation):
        super().__init__()
        activations = {"relu": nn.ReLU, "tanh": nn.Tanh}
        if activation not in activations:
            raise ValueError(f"Unknown activation: {activation}")

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)

        layers = []
        in_features = context_length * embedding_dim
        for hidden_size in hidden_layers:
            layers += [nn.Linear(in_features, hidden_size), activations[activation]()]
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            in_features = hidden_size
        layers.append(nn.Linear(in_features, vocab_size))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = self.embedding(x).flatten(start_dim=1)  # (batch, context_length * embedding_dim)
        return self.network(x)  # raw logits, shape (batch, vocab_size)

## 6. Training helper

In [8]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total_examples = 0.0, 0
    with torch.set_grad_enabled(training):
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            loss = criterion(model(X_batch), y_batch)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * X_batch.size(0)
            total_examples += X_batch.size(0)
    return total_loss / total_examples

## 7. Hyperparameter search

In [9]:
GRID = {
    "embedding_dim": [16, 32, 64],
    "hidden_layers": [[128], [256, 128], [256, 128, 64], [256, 128, 64, 32]],
    "dropout_rate": [0.1, 0.2, 0.3],
    "activation": ["tanh"],
}
MAX_EPOCHS = 50
PATIENCE = 10
LEARNING_RATE = 0.01

criterion = nn.CrossEntropyLoss()
results = []
best = {"val_loss": float("inf")}

for values in product(*GRID.values()):
    config = dict(zip(GRID.keys(), values))
    torch.manual_seed(SEED)

    model = SurnameGeneratorModel(
        vocab_size=vocab_size, context_length=CONTEXT_LENGTH, **config
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_val, best_train, best_epoch, best_weights = float("inf"), None, 0, None
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = run_epoch(model, train_loader, criterion, optimizer)
        val_loss = run_epoch(model, val_loader, criterion)
        if val_loss < best_val:
            best_val, best_train, best_epoch = val_loss, train_loss, epoch
            best_weights = copy.deepcopy(model.state_dict())
        elif epoch - best_epoch >= PATIENCE:
            break

    results.append({**config, "val_loss": best_val,
                    "train_loss": best_train, "best_epoch": best_epoch})
    print(f"{config} -> val {best_val:.4f}, train {best_train:.4f} (epoch {best_epoch})")

    if best_val < best["val_loss"]:
        best = {"val_loss": best_val, "config": config, "weights": best_weights}

{'embedding_dim': 16, 'hidden_layers': [128], 'dropout_rate': 0.1, 'activation': 'tanh'} -> val 1.9076, train 1.7935 (epoch 2)
{'embedding_dim': 16, 'hidden_layers': [128], 'dropout_rate': 0.2, 'activation': 'tanh'} -> val 1.9107, train 1.8268 (epoch 2)
{'embedding_dim': 16, 'hidden_layers': [128], 'dropout_rate': 0.3, 'activation': 'tanh'} -> val 1.8953, train 1.6372 (epoch 5)
{'embedding_dim': 16, 'hidden_layers': [256, 128], 'dropout_rate': 0.1, 'activation': 'tanh'} -> val 1.9809, train 1.7690 (epoch 8)
{'embedding_dim': 16, 'hidden_layers': [256, 128], 'dropout_rate': 0.2, 'activation': 'tanh'} -> val 1.9277, train 1.8376 (epoch 8)
{'embedding_dim': 16, 'hidden_layers': [256, 128], 'dropout_rate': 0.3, 'activation': 'tanh'} -> val 1.9236, train 1.8923 (epoch 9)
{'embedding_dim': 16, 'hidden_layers': [256, 128, 64], 'dropout_rate': 0.1, 'activation': 'tanh'} -> val 1.9102, train 1.8355 (epoch 29)
{'embedding_dim': 16, 'hidden_layers': [256, 128, 64], 'dropout_rate': 0.2, 'activatio

In [10]:
results.sort(key=lambda r: r["val_loss"])
for rank, r in enumerate(results[:10], start=1):
    print(f"{rank:>2}. emb={r['embedding_dim']:<3} layers={str(r['hidden_layers']):<18} "
          f"dropout={r['dropout_rate']}  val={r['val_loss']:.4f}  epoch={r['best_epoch']}")

print("\nBest configuration:", best["config"])
print(f"Best validation loss: {best['val_loss']:.4f}")

 1. emb=16  layers=[256, 128, 64]     dropout=0.2  val=1.8932  epoch=29
 2. emb=16  layers=[128]              dropout=0.3  val=1.8953  epoch=5
 3. emb=16  layers=[256, 128, 64]     dropout=0.3  val=1.9044  epoch=35
 4. emb=16  layers=[128]              dropout=0.1  val=1.9076  epoch=2
 5. emb=16  layers=[256, 128, 64]     dropout=0.1  val=1.9102  epoch=29
 6. emb=16  layers=[128]              dropout=0.2  val=1.9107  epoch=2
 7. emb=32  layers=[256, 128, 64]     dropout=0.1  val=1.9171  epoch=43
 8. emb=16  layers=[256, 128]         dropout=0.3  val=1.9236  epoch=9
 9. emb=32  layers=[256, 128, 64]     dropout=0.2  val=1.9243  epoch=21
10. emb=32  layers=[128]              dropout=0.3  val=1.9259  epoch=4

Best configuration: {'embedding_dim': 16, 'hidden_layers': [256, 128, 64], 'dropout_rate': 0.2, 'activation': 'tanh'}
Best validation loss: 1.8932


## 8. Save and reload the best model

In [11]:
torch.save({
    "model_state_dict": best["weights"],
    "config": best["config"],
    "context_length": CONTEXT_LENGTH,
    "vocab": vocab,
    "validation_loss": best["val_loss"],
}, MODEL_PATH)

checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=True)
best_model = SurnameGeneratorModel(
    vocab_size=len(checkpoint["vocab"]),
    context_length=checkpoint["context_length"],
    **checkpoint["config"],
).to(device)
best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()

SurnameGeneratorModel(
  (embedding): Embedding(42, 16, padding_idx=2)
  (network): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): Tanh()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): Tanh()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): Tanh()
    (8): Dropout(p=0.2, inplace=False)
    (9): Linear(in_features=64, out_features=42, bias=True)
  )
)

## 9. Generating surnames

In [12]:
@torch.no_grad()
def next_char_probs(prefix="", temperature=1.0):
    """Probability of each token being the next one after `prefix`."""
    ids = [START_ID] + [char_to_id[ch] for ch in prefix.lower()]
    if len(ids) >= CONTEXT_LENGTH:
        raise ValueError("Prefix is too long for the model's context")
    x = torch.tensor([pad_right(ids)], dtype=torch.long, device=device)
    logits = best_model(x)[0] / temperature
    logits[[START_ID, PAD_ID]] = float("-inf")
    return F.softmax(logits, dim=0)


def generate_surname(prefix="", temperature=1.0):
    name = prefix.lower()
    while len(name) + 1 < CONTEXT_LENGTH:  # +1 for <START>
        probs = next_char_probs(name, temperature)
        next_id = torch.multinomial(probs, num_samples=1).item()
        if next_id == END_ID:
            break
        name += id_to_char[next_id]
    return name


def inspect_next_char(prefix, top_k=10):
    top_p, top_ids = torch.topk(next_char_probs(prefix), top_k)
    pairs = ", ".join(f"{id_to_char[i]} {p:.3f}" for p, i in zip(top_p.tolist(), top_ids.tolist()))
    print(f"After '{prefix}': {pairs}")

In [13]:
for temperature in (0.5, 1.0, 1.2):
    print(f"prefix 'սար', temperature {temperature}:",
          [generate_surname("սար", temperature) for _ in range(5)])
print("no prefix:", [generate_surname() for _ in range(10)])

for prefix in ("", "ս", "սար"):
    inspect_next_char(prefix)

prefix 'սար', temperature 0.5: ['սարույան', 'սարոնյան', 'սարաւաննյյան', 'սարյան', 'սարոււկյան']
prefix 'սար', temperature 1.0: ['սարեյան', 'սարաբյան', 'սարիլակյեն', 'սարգսխացլյան', 'սարրաբյան']
prefix 'սար', temperature 1.2: ['սարֆոլնյլա', 'սարռալյանց', 'սարդւոսսյան', 'սարաւնմյան', 'սարրոյանն']
no prefix: ['քաճաւակկյւրն', 'զողաւչյան', 'նամտյան', 'աղեխյան', 'կաքամյան', 'տորջտայան', 'ջալաղկյման', 'եդբգալյան', 'խամրանյան', 'սաջիգււնյան']
After '': ա 0.258, բ 0.090, թ 0.086, մ 0.078, ս 0.065, գ 0.050, դ 0.048, խ 0.041, ե 0.038, ի 0.025
After 'ս': ա 0.616, ո 0.128, ի 0.101, ե 0.039, ս 0.014, դ 0.013, յ 0.012, ն 0.009, տ 0.009, բ 0.008
After 'սար': ա 0.336, ո 0.204, ի 0.081, յ 0.048, տ 0.045, ե 0.040, ս 0.031, ր 0.027, խ 0.026, դ 0.023


### Sanity check: are the generated names new?

In [14]:
generated = [generate_surname() for _ in range(500)]
known = set(surnames)
print(f"Distinct names: {len(set(generated))} / {len(generated)}")
print(f"Already in the dataset: {sum(g in known for g in generated) / len(generated):.1%}")

Distinct names: 498 / 500
Already in the dataset: 0.6%


In [21]:
for i in range(5):
    print(generate_surname())

աեհզաշյան
դասոլանյևն
աքոֆտսյան
սաոկյան
ադեւյան


## Notes and limitations

- The dataset is small (a few hundred surnames), and validation loss stops improving after only a few epochs, so the models overfit quickly.
- The validation set is used both to choose the configuration and to report its loss. A separate test set would give an unbiased final number.
- The model reads a fixed-size window with separate weights for each position. A recurrent or transformer model, or a shorter sliding window, would share what it learns across positions.
- The surname file is not included in the notebook. Readers need their own copy at `../data/surnames_hy.txt`.